In [ ]:
import sys
import os

# Ensure repository root is on sys.path so `core` is importable when running this notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Imports — KalKalori BareTubeHeatExchanger test

from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle

from core.heat_transfer.internal_flow import (
    FluidProps as TubeFluidProps,
)

from core.heat_transfer.outside_flow import (
    FluidProps as OutsideFluidProps,
)

from core.heat_transfer.streams import (
    SensibleHeatStream,
)

from core.models.bare_tube import (
    BareTubeHeatExchanger,
)

In [ ]:
from core.heat_transfer.outside_flow import (
    FluidProps,
    check_outside_ht_applicability,
    outside_flow_from_mass_flow,
)

In [ ]:
props_nominal = FluidProps(
    rho=1.16,
    mu=2.0e-5,
    k=0.03,
    cp=1026.4,
)

props_low_pr = FluidProps(
    rho=1.16,
    mu=2.0e-5,
    k=0.08,
    cp=150.0,
)

props_high_pr = FluidProps(
    rho=1.16,
    mu=0.02,
    k=0.02,
    cp=1000.0,
)


In [ ]:
def run_solver_case(name, props, **kwargs):
    v, Re, Pr, alfa_o, dp_o, warnings_list = outside_flow_from_mass_flow(
        props=props,
        **kwargs,
    )
    return {
        'name': name,
        'v': v,
        'Re': Re,
        'Pr': Pr,
        'alfa_o': alfa_o,
        'dp_o': dp_o,
        'warnings': warnings_list,
    }


def run_check_case(name, Re, Pr, D, ST, SL, layout, n_rows, use_vmax_for_ht=True):
    warnings_list = check_outside_ht_applicability(
        Re=Re,
        Pr=Pr,
        tube_outer_diameter=D,
        tube_pitch_transverse=ST,
        tube_pitch_longitudinal=SL,
        layout=layout,
        n_rows=n_rows,
        use_vmax_for_ht=use_vmax_for_ht,
    )
    return {
        'name': name,
        'Re': Re,
        'Pr': Pr,
        'warnings': warnings_list,
    }


In [ ]:
cases = []

cases.append(run_solver_case(
    'nominal',
    props_nominal,
    m_dot=5.5,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=4,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'very_low_re',
    props_nominal,
    m_dot=0.03,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=4,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'high_re',
    props_nominal,
    m_dot=800.0,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=4,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'staggered_tight_pitch',
    props_nominal,
    m_dot=5.5,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.020,
    layout='staggered',
    n_rows=3,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'single_row',
    props_nominal,
    m_dot=5.5,
    frontal_area=0.133,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=1,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'low_pr',
    props_low_pr,
    m_dot=5.5,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=4,
    n_tubes_per_row=20,
))

cases.append(run_solver_case(
    'high_pr',
    props_high_pr,
    m_dot=5.5,
    frontal_area=0.532,
    tube_outer_diameter=0.018,
    tube_pitch_transverse=0.025,
    tube_pitch_longitudinal=0.025,
    layout='inline',
    n_rows=4,
    n_tubes_per_row=20,
))

cases.append(run_check_case(
    'geometry_invalid_st_blockage',
    Re=5000.0,
    Pr=0.7,
    D=0.018,
    ST=0.018,
    SL=0.025,
    layout='inline',
    n_rows=4,
))

cases.append(run_check_case(
    'geometry_near_blockage',
    Re=5000.0,
    Pr=0.7,
    D=0.018,
    ST=0.019,
    SL=0.019,
    layout='inline',
    n_rows=4,
))

cases.append(run_check_case(
    'geometry_large_pitch',
    Re=5000.0,
    Pr=0.7,
    D=0.018,
    ST=0.09,
    SL=0.09,
    layout='inline',
    n_rows=4,
))

cases.append(run_check_case(
    'not_using_vmax_for_ht',
    Re=5000.0,
    Pr=0.7,
    D=0.018,
    ST=0.025,
    SL=0.025,
    layout='inline',
    n_rows=4,
    use_vmax_for_ht=False,
))


In [ ]:
for case in cases:
    print('=' * 80)
    print(case['name'])
    if 'v' in case:
        print(f"v       = {case['v']:.6f} m/s")
    print(f"Re      = {case['Re']:.6f}")
    print(f"Pr      = {case['Pr']:.6f}")
    if 'alfa_o' in case:
        print(f"alfa_o  = {case['alfa_o']:.6f} W/(m^2*K)")
    if 'dp_o' in case:
        print(f"dp_o    = {case['dp_o']:.6f} Pa")
    print('warnings:')
    if case['warnings']:
        for w in case['warnings']:
            print(f"  - {w}")
    else:
        print('  (none)')
    print()


In [ ]:
unique_warnings = sorted(
    {w for case in cases for w in case['warnings']},
    key=lambda warning: str(warning),
)
print(f'unique warnings: {len(unique_warnings)}')
for w in unique_warnings:
    print('-', w)
